In [ ]:
from google.colab import files

files.download("/content/app.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
### buat repository github
#1. Buka  ⁠GitHub.
#2. Login
#3. Buat repository baru dengan nama "evdm-bivariat"
#4. Pilih Public agar mudah digunakan untuk latihan.
#5. Buat repository.
#6. Unggah file app.py yang telah diunduh dari Google Colab ke repository tersebut
#7. Buat file baru bernama requirements.txt pada repository yang sama
#8. Isi file requiremenments.txt dengan:
    streamlit
    pandas
    plotly
#9. Deploy aplikasi:
    #1) Buka  ⁠streamlit Community Cloud.
    #2) Login dan hubungkan akun GitHub.
    #3) Pilih Create app.
    #4) Pilih repository restaurant-tips-dashboard.
    #5) Pilih branch main.
    #6) Isi main file path dengan app.py.
    #7) Klik Deploy.

IndentationError: unexpected indent (2751327211.py, line 10)

In [ ]:
!pip install -q streamlit pandas seaborn plotly pillow

In [ ]:
import pandas as pd
import seaborn as sns

# Memuat dataset Iris
df = sns.load_dataset("iris")

# Menampilkan lima observasi pertama
display(df.head())

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [ ]:
print("Ukuran dataset:", df.shape)

print("\nNama variabel:")
print(df.columns.tolist())

print("\nInformasi data:")
df.info()

Ukuran dataset: (150, 5)

Nama variabel:
['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

Informasi data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
 4   species       150 non-null    object 
dtypes: float64(4), object(1)
memory usage: 6.0+ KB


In [ ]:
df.describe()

,sepal_length,sepal_width,petal_length,petal_width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.057333,3.758000,1.199333
std,0.828066,0.435866,1.765298,0.762238
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


In [ ]:
%%writefile app.py

import streamlit as st
import pandas as pd
import seaborn as sns
import plotly.express as px

# KONFIGURASI HALAMAN

st.set_page_config(
    page_title="Iris Dashboard",
    page_icon="📊",
    layout="wide"
)

# JUDUL DASHBOARD

st.title("My Iris Dataset Dashboard 🌺")

st.write(
    """
    Dashboard ini digunakan untuk mengeksplorasi
    hubungan antarvariabel pada dataset Iris melalui
    visualisasi data interaktif.
    """
)

st.divider()

# MEMUAT DATASET

@st.cache_data
def load_data():
    return sns.load_dataset("iris")

df = load_data()

# Mendefinisikan atribut dan spesies
attributes = df.columns[:-1].tolist()
species = df["species"].unique().tolist()

# INFORMASI DATASET

with st.expander("About the Iris Dataset"):

    st.header("About the Iris Dataset")

    st.write(
        """
        The Iris dataset consists of 150 observations
        from three species of iris flowers: Setosa,
        Versicolor, and Virginica.

        Each observation contains four numerical
        attributes: sepal length, sepal width,
        petal length, and petal width.

        The dataset is commonly used to demonstrate
        exploratory data analysis, data visualization,
        and classification.
        """
    )

# SIDEBAR: PILIHAN VARIABEL DAN SPESIES

with st.sidebar:

    st.header("Filters")

    # Pilihan atribut
    selected_attribute = st.selectbox(
        "Select Attribute:",
        attributes,
        index=0
    )

    # Pilihan spesies
    selected_species = st.multiselect(
        "Select Species:",
        species,
        default=species,
        placeholder="Filter by species"
    )

# Memfilter data berdasarkan spesies yang dipilih
filtered_df = df[
    df["species"].isin(selected_species)
]

# STATISTIK DESKRIPTIF

st.header("Basic Statistics")

st.write(
    "Statistik deskriptif untuk empat variabel numerik."
)

st.dataframe(
    filtered_df[attributes].describe(),
    use_container_width=True
)

# FUNGSI VISUALISASI PAIRWISE SCATTER PLOT

def create_pairplot(data, attributes):

    fig = px.scatter_matrix(
        data,
        dimensions=attributes,
        color="species",
        opacity=0.6,
        title="Pairwise Scatter Plot"
    )

    fig.update_traces(
        diagonal_visible=False
    )

    fig.update_layout(
        height=750
    )

    return fig

# FUNGSI HISTOGRAM

def create_histogram(data, attribute):

    fig = px.histogram(
        data,
        x=attribute,
        color="species",
        marginal="box",
        barmode="overlay",
        opacity=0.7,
        title=f"Histogram of {attribute}"
    )

    fig.update_layout(
        hovermode="x unified"
    )

    return fig

# FUNGSI VIOLIN PLOT

def create_violin_plot(
    data,
    attribute,
    show_points=True
):

    points = "all" if show_points else False

    fig = px.violin(
        data,
        x="species",
        y=attribute,
        color="species",
        box=True,
        points=points,
        title=f"Violin Plot of {attribute}"
    )

    return fig

# VISUALISASI BIVARIAT: SCATTER MATRIX

st.header("Data Visualization")

st.subheader(
    "Pairwise Scatter Plot (All Attributes)"
)

st.write(
    """
    Scatter matrix digunakan untuk mengeksplorasi
    hubungan setiap pasangan variabel numerik.
    Warna menunjukkan spesies bunga.
    """
)

if filtered_df.empty:

    st.warning(
        "Pilih minimal satu spesies pada sidebar."
    )

else:

    pairplot_fig = create_pairplot(
        filtered_df,
        attributes
    )

    st.plotly_chart(
        pairplot_fig,
        use_container_width=True
    )

# VISUALISASI DISTRIBUSI ATRIBUT

st.subheader("Single Attribute Distribution")

st.write(
    """
    Pilih satu atribut untuk membandingkan distribusinya
    pada ketiga spesies bunga.
    """
)

if not filtered_df.empty:

    col1, col2 = st.columns(2)

    # Histogram
    with col1:

        hist_fig = create_histogram(
            filtered_df,
            selected_attribute
        )

        st.plotly_chart(
            hist_fig,
            use_container_width=True
        )

    # Violin plot
    with col2:

        show_points = st.checkbox(
            "Show individual points",
            value=True
        )

        violin_fig = create_violin_plot(
            filtered_df,
            selected_attribute,
            show_points
        )

        st.plotly_chart(
            violin_fig,
            use_container_width=True
        )

# TABEL DATA

with st.expander("View Dataset"):

    st.dataframe(
        filtered_df,
        use_container_width=True
    )

st.divider()

st.caption(
    "Praktikum Eksplorasi dan Visualisasi Data Bivariat"
)

Writing app.py


In [ ]:
import pandas as pd
import seaborn as sns
import plotly.express as px

# Memuat dataset Iris
df = sns.load_dataset("iris")

# Memilih variabel numerik
attributes = df.select_dtypes(include="number").columns.tolist()

# Membuat scatter matrix
fig = px.scatter_matrix(
    df,
    dimensions=attributes,
    color="species",
    opacity=0.6,
    title="Pairwise Scatter Plot Dataset Iris"
)

# Menyembunyikan grafik pada diagonal
fig.update_traces(diagonal_visible=False)

# Menampilkan grafik
fig.show()

In [ ]:
fig = px.histogram(
    df,
    x="sepal_length",
    color="species",
    marginal="box",
    barmode="overlay",
    opacity=0.7,
    title="Distribusi Panjang Sepal Berdasarkan Spesies",
    labels={
        "sepal_length": "Panjang Sepal (cm)",
        "species": "Spesies"
    }
)

fig.show()

In [ ]:
fig = px.violin(
    df,
    x="species",
    y="petal_length",
    color="species",
    box=True,
    points="all",
    title="Distribusi Panjang Petal Berdasarkan Spesies",
    labels={
        "species": "Spesies",
        "petal_length": "Panjang Petal (cm)"
    }
)

fig.show()